In [ ]:
import os
import rasterio
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.cluster import MiniBatchKMeans


from sklearn.preprocessing import StandardScaler

In [ ]:
rundir = "/home/ebr/projects/release-volume-sampler/generated/messina_001_20240819_093931"

n_clusters = 1000
weights = [20,20,1,1,1]

In [ ]:

features = {}
for filename in filter(lambda file: file.endswith(".tif"), os.listdir(rundir)): 
    with rasterio.open(os.path.join(rundir,filename)) as raster:
        print(raster.name)

In [ ]:
# Load files
with rasterio.open(os.path.join(rundir, "slope.tif")) as ds:
    slope = ds.read(1)

with rasterio.open(os.path.join(rundir, "aspect.tif")) as ds:
    aspect = ds.read(1)

with rasterio.open(os.path.join(rundir, "bathy.tif")) as bathy_ds:
    z = bathy_ds.read(1)
    bathy_transform = bathy_ds.transform


# Create features for labeling
cols, rows = np.meshgrid(np.arange(bathy_ds.width), np.arange(bathy_ds.height))
xs, ys = bathy_transform * (rows, cols)

sin_aspect = np.sin(aspect*np.pi/180.)
cos_aspect = np.cos(aspect*np.pi/180.)

mask = z<0
X = np.vstack([xs[mask], ys[mask], slope[mask], sin_aspect[mask], cos_aspect[mask]]).T

scaler = StandardScaler()
scaler.fit(X)

X_trans = np.matmul(scaler.transform(X), np.diag(weights))

In [ ]:
# Fit the labels
#kmeans = KMeans(n_clusters=200, random_state=0, n_init="auto", algorithm="lloyd")
kmeans = MiniBatchKMeans(n_clusters=1000, max_no_improvement=20, batch_size=256*10) # This is much faster
kmeans.fit(X_trans)

In [ ]:
# Fit the labels
weights = [20,20,1,1,1]
kmeans.fit(np.matmul(X_trans,np.diag(weights)))

In [ ]:
# Get the labels
labels = np.empty(bathy_ds.shape)
labels[:] = np.nan
labels[mask] = kmeans.labels_

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(labels)

In [ ]:
# write to file.

with rasterio.open(
    os.path.join(rundir, "labels.tif"),
    'w',
    driver='GTiff',
    height=labels.shape[0],
    width=labels.shape[1],
    count=1,
    dtype=labels.dtype,
    crs='+proj=latlong',
    transform=bathy_ds.transform,
) as dst:
    dst.write(labels, 1)

In [ ]:
df = pd.DataFrame(X, columns=["xs", "ys", "slope", "sin_aspect", "cos_aspect"])
df["labels"] = kmeans.labels_

In [ ]:
df

In [ ]:
sin_aspect_by_label = {f"label_{i}": df[df.labels == i].sin_aspect.values for i in range(200)}
cos_aspect_by_label = {f"label_{i}": df[df.labels == i].cos_aspect.values for i in range(200)}
slope_by_label = {f"label_{i}": df[df.labels == i].slope.values for i in range(200)}

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, layout='constrained')
fig.set_size_inches(18, 8)

patches = slice(0,100)

ax1.boxplot(list(slope_by_label.values())[patches], showfliers=False)
ax1.set_label("Slope")

ax2.boxplot(list(sin_aspect_by_label.values())[patches], showfliers=False)
ax2.set_label("sin(aspect)")

ax3.boxplot(list(cos_aspect_by_label.values())[patches], showfliers=False)
ax3.set_label("cos(aspect)")

plt.show()

# Picking random volumes

We are interested in applying the partitioning to select potential volumes. As a start we select a random patch and add a specified number of neighbours.

In [ ]:
centers_df = pd.DataFrame(kmeans.cluster_centers_, columns=["xs", "ys", "slope", "sin_aspect", "cos_aspect"])
plt.scatter(centers_df["xs"], centers_df["ys"])